In [1]:
include("code.jl")

Code correctly loaded.


## Permanent approximation  

Computing the permanent is known to be a hard task, thus using the best known algorithm (Ryser algorithm) is not the best choice. We can use approximate algorithm like Gurvits, which are able to approximate $\operatorname{perm}(A)$ up to erro $\epsilon \lVert A\rVert^{n}$ in $\mathcal{O}(n^{2}/\epsilon^{2})$. Notice that the error is exponential in the largest eigenvalue. Nevertheless to compute the characteristic function we just need to compute permanents of unitary matrix, thus the largest eigenvalue will always have modulus $1$, making the algorithm able to give us additive error on the characteristic function.

In [2]:
n=10
k=2
U=RandHaar(n).U
A=U'*diagm([ones(Int, k); zeros(Int, n - k)])*U

Perm=ryser(A)
Estimated_Perm=estimate_permanent(A)
println("Permanent:           ", Perm)
println("Estimated Permanent: ", Estimated_Perm)
println("Absolute Error:      ", abs(Perm - Estimated_Perm))

Permanent:           4.4916417562833846e-5 - 5.61159327555974e-20im
Estimated Permanent: 5.552228105639272e-5 - 1.266258839161337e-5im
Absolute Error:      1.651742974010863e-5


To compute the output probability distribution of a boson sampling machine, we can compute 
\begin{equation}
    \chi(\phi_{1},...,\phi_{m}|U)=\langle\Psi_{in}|U^{\dagger}e^{i\sum_{j}\phi_{j}\hat{n}_{j}}U|\Psi_{in}\rangle=\operatorname{perm}\left(U^{\dagger}\Phi U\right)
\end{equation}
with $\Phi=\operatorname{diag}(\phi_{1},...,\phi_{m})$. In the case we consider partial distinguishable particles, we can take care of it by adding the distinguishability matrix $S$ as follows
\begin{equation}
    \chi(\phi_{1},...,\phi_{m}|U,S)=\operatorname{perm}\left(U^{\dagger}\Phi U\odot S\right)
\end{equation}
which reduces to the above in the case on indistinguishable particles, since it correspond to $S_{ij}=1\forall i,j$.

In the following we are interested in the two mode correlators, which can be expressed as
\begin{align}
    \langle \hat{n}_{i}\hat{n}_{j}\rangle &= \delta_{ij} \sum_{k=1}^n |U_{ik}|^2 + \sum_{k \neq l} \left( |U_{ik}|^2 |U_{jl}|^2 + |S_{kl}|^2 U_{ik}^* U_{il} U_{jl}^* U_{jk} \right) \quad .
\end{align}
Notice that if we are interested to a system with internal degrees of freedom which are mixed states we can just substitute $\operatorname{Tr}[\rho_{i}\rho_{j}]$ to the term $ |S_{ij}|^2$.

## Obtaining the Characteristic function 

We consider two copies of our system which evolves through a linear unitary $U$ and then we apply virtual distillation. The overall matrix to diagonalize to compute the characteristic function is
\begin{equation}
    U_{tot}=e^{i\sum_{j}\phi_{j}\hat{n}_{j,1}}\hat{S}_{2}(U\oplus U)
\end{equation}

In [3]:
m=2

U=RandHaar(m).U
ϕ=rand(m)

V=U_tot(U,ϕ)

Perm=ryser(V)
Estimated_Perm=CF(ϕ,U,1;Samples=10^6)
println("CF (Ryser)    : ", Perm)
println("CF (Gurvits)  : ", Estimated_Perm)
println("Absolute Error: ", abs(Perm - Estimated_Perm))

CF (Ryser)    : 0.8369139458181081 + 0.45374234924590107im
CF (Gurvits)  : 0.8370359618649109 + 0.4535172943790249im
Absolute Error: 0.00025600314213315885


## Expectation value from the derivatives   

Let us consider now thermal states.

In [40]:
m=6
U=RandHaar(m).U;
abs2.(U)

6×6 Matrix{Float64}:
 0.200924   0.087319   0.417769   0.00318826  0.26242     0.0283791
 0.16997    0.083211   0.184164   0.234232    0.036785    0.291637
 0.225095   0.111989   0.152249   0.292769    0.0375229   0.180376
 0.0737807  0.633121   0.0813822  0.0278234   0.00503286  0.17886
 0.0939612  0.072673   0.138101   0.0357617   0.650887    0.0086167
 0.236269   0.0116875  0.0263343  0.406226    0.00735217  0.312131

In [44]:
β=2*atanh.(collect(LinRange(0.65,0.99,20)))
Number_of_samples=10^6
δ= 0.01

y_n=[compute_density_correlation(1,2, U, (1-tanh(x/2))*Matrix(I,m,m)+tanh(x/2)*ones(m,m)) for x in β]
y_d=[calc_n1n2_thermal(δ,U,x;num_samples=Number_of_samples) for x in β];

In [67]:
# Fit a quadratic polynomial to the data
x = tanh.(β ./ 2)
y = y_d

# 2. Construct the design matrix A = [1  x  x^2]
A = [ones(length(x)) x x.^2]

# 3. Solve for the coefficients [a_0, a_1, a_2] using least squares
coeffs = A \ y

println("Coefficients (a_0, a_1, a_2): ", coeffs)

# 4. Calculate the fitted values
y_fitted = A * coeffs




Coefficients (a_0, a_1, a_2): [0.8066102896325297, -0.15617597573235245, 0.07493038843397737]


20-element Vector{Float64}:
 0.736753994519856
 0.7357263783196912
 0.7347467507782686
 0.7338151118955882
 0.7329314616716501
 0.7320958001064541
 0.7313081272000004
 0.7305684429522888
 0.7298767473633194
 0.7292330404330922
 0.7286373221616073
 0.7280895925488646
 0.727589851594864
 0.7271380992996057
 0.7267343356630895
 0.7263785606853157
 0.726070774366284
 0.7258109767059944
 0.7255991677044471
 0.725435347361642

In [45]:
Ideal=compute_density_correlation(1,2, U,ones(m,m))*ones(length(β))
y_m=[compute_density_correlation(1,2, U, (1-tanh(x))*Matrix(I,m,m)+tanh(x)*ones(m,m)) for x in β];

In [70]:

# Pre-calculate data
x_data = tanh.(β ./ 2)
ideal_y = Ideal .* ones(length(β))

# Apply global styles mirroring the PDF
default(
    fontfamily="serif",
    framestyle=:box,                 
    grid=false,                      
    tick_direction=:in,              
    legend=:topright,      
    foreground_color_legend=nothing, 
    dpi=600,
    markerstrokecolor=:auto          # <-- This forces the stroke to match the line/fill color
)

mark=7

# 1. Use `scatter` instead of `plot` to plot ONLY markers (no lines)
scatter(x_data, y_n, label="Noisy", xlabel=L"\mathrm{Tr}[\rho^2]", ylabel=L"\langle n_1 n_2 \rangle", marker=:+, markersize=mark,color="#0F1F7A")
scatter!(x_data, y_m, label="Half temperature", marker=:+, markersize=mark,color="#0DA0A3")
scatter!(x_data, y_fitted, label="M=2", marker=:+, markersize=mark,color="#C33225")

# 2. Keep `plot!` for the Ideal line since you want a dashed line, not markers
plot!(x_data, ideal_y, label="Ideal", linestyle=:dash, color="#939393", linewidth=1.5)
savefig("Correlator_Distillation.pdf")

sys:1: UserWarning: You passed a edgecolor/edgecolors ((0.058823529411764705, 0.12156862745098039, 0.47843137254901963, 1.0)) for an unfilled marker ('+').  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
sys:1: UserWarning: No data for colormapping provided via 'c'. Parameters 'vmin', 'vmax' will be ignored
sys:1: UserWarning: You passed a edgecolor/edgecolors ((0.050980392156862744, 0.6274509803921569, 0.6392156862745098, 1.0)) for an unfilled marker ('+').  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
sys:1: UserWarning: You passed a edgecolor/edgecolors ((0.7647058823529411, 0.19607843137254902, 0.1450980392156863, 1.0)) for an unfilled marker ('+').  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.


"c:\\Users\\admir\\Desktop\\PhD\\Open Projects\\Virtual distillation\\notebooks\\Distillation of Boson Sampling\\Correlator_Distillation.pdf"

Coefficients (a_0, a_1, a_2): [0.14000000000000654, -0.1114285714285762, 1.0285714285714294]


5-element Vector{Float64}:
  1.0571428571428596
  4.031428571428571
  9.06285714285714
 16.15142857142857
 25.29714285714286